<a href="https://colab.research.google.com/github/Yogskiy/big-data-kelompok-7-weather/blob/main/Assessing_Cleaning_Data_Cuaca_Jabar_Kelompok7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Wrangling — Assessing & Cleaning Data
## Analisis dan Prediksi Pola Cuaca Jawa Barat untuk Mitigasi Banjir dan Dukungan Sektor Pertanian

**Mata Kuliah:** Big Data  
**Kelompok:** 7  
**Anggota:**
1. Hilarius Yoga Baskoro — 101032300159
2. Wildan Janitra Ramadhan — 101032300140

**Tahap 2:** *Data Assessing* (Pemeriksaan Kualitas Data) & *Data Cleaning* (Pembersihan dan Standarisasi Data)

> Notebook ini melanjutkan hasil dari tahap **Gathering Data** (`Gathering_Data_Cuaca_Jabar_Kelompok7.ipynb`). Proses dibagi secara tegas menjadi dua fase berurutan:
> 1. **Data Assessing:** Mengaudit seluruh 210.528 baris × 26 kolom secara mendalam (kelengkapan nilai, variansi kolom, tipe data semantik, konsistensi hukum fisika/meteorologi antar-kolom, serta distribusi pencilan/*outliers*) **tanpa mengubah isi data**.
> 2. **Data Cleaning:** Mengeksekusi perbaikan terukur berdasarkan bukti temuan pada tahap *assessing*, memverifikasi ulang kualitas akhir (*post-cleaning quality gate*), lalu menyimpan dataset bersih ke `cuaca-jabar/03_dataset_clean/` agar siap digunakan pada tahap *Exploratory Data Analysis (EDA)* dan pemodelan Machine Learning.

---
## 1. Kerangka Metodologi Assessing & Cleaning

Dalam pipeline Big Data, kesalahan pada tahap pembersihan data dapat merusak sinyal prediktif yang justru dibutuhkan oleh kasus penggunaan utama kita:
- **Mitigasi Risiko Banjir:** Sangat bergantung pada akurasi variabel hidrologi (`precipitation`, `rain`, `showers`, `precipitation_probability`) terutama pada ekor distribusi kanan (*extreme rainfall events*).
- **Dukungan Agrikultur Presisi:** Sangat bergantung pada konsistensi termodinamika antara suhu udara (`temperature_2m`), titik embun (`dew_point_2m`), kelembapan (`relative_humidity_2m`), defisit tekanan uap (`vapour_pressure_deficit` / VPD), serta suhu tanah berlapis (`0cm`, `6cm`, `18cm`).

Oleh sebab itu, tahap **Assessing** dirancang untuk menguji **6 dimensi kualitas data**:

| No | Dimensi Kualitas | Pertanyaan Audit Utama |
| :-: | --- | --- |
| 1 | **Validity & Semantic Types** | Apakah tipe data setiap kolom sudah sesuai maknanya (mis. `time` sebagai `datetime64`, `weather_code` sebagai kategorik)? |
| 2 | **Completeness** | Kolom mana yang memiliki *missing values* (`NaN`), berapa proporsinya, dan apakah polanya acak (*MCAR*) atau sistematis menurut waktu/kota (*MAR/MNAR*)? |
| 3 | **Uniqueness & Variance** | Apakah ada baris duplikat pada kunci `(city, time)` atau kolom bernilai konstan (*zero variance*) yang tidak informatif? |
| 4 | **Physical & Domain Consistency** | Apakah hubungan antar-variabel mematuhi hukum fisika meteorologi (mis. `precipitation == rain + showers` dan `dew_point_2m <= temperature_2m`)? |
| 5 | **Spatial Grid Integrity** | Bagaimana perilaku *grid snapping* koordinat model Open-Meteo terhadap 12 titik kota/kabupaten Jawa Barat? |
| 6 | **Outliers & Extreme Events** | Mana nilai ekstrem yang merupakan kesalahan data (*noise*) dan mana yang merupakan fenomena cuaca ekstrem nyata yang wajib dipertahankan? |

---
## 2. Import Library & Pemuatan Dataset Hasil Gathering

Dataset yang dimuat adalah berkas keluaran tahap *Gathering* dari folder `cuaca-jabar/02_dataset_final/`. Sel penyiapan direktori dibuat adaptif agar dapat dijalankan baik di **Google Colab** (terhubung ke Google Drive) maupun di **Jupyter lokal**.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print("pandas :", pd.__version__)
print("numpy  :", np.__version__)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE_DIR = Path("/content/drive/MyDrive/cuaca-jabar")
except Exception as e:
    print(f"Google Drive tidak tersedia ({type(e).__name__}), memakai penyimpanan lokal.")
    BASE_DIR = Path("./cuaca-jabar")

FINAL_DIR = BASE_DIR / "02_dataset_final"   # keluaran Tahap 1 (Gathering)
CLEAN_DIR = BASE_DIR / "03_dataset_clean"   # keluaran Tahap 2 (Cleaning)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print("Sumber dataset (Stage 1):", FINAL_DIR)
print("Tujuan dataset (Stage 2):", CLEAN_DIR)

pandas : 2.2.3
numpy  : 2.0.2
Google Drive tidak tersedia (ModuleNotFoundError), memakai penyimpanan lokal.
Sumber dataset (Stage 1): cuaca-jabar/02_dataset_final
Tujuan dataset (Stage 2): cuaca-jabar/03_dataset_clean


In [2]:
path_csv_raw = FINAL_DIR / "cuaca_jabar_12kota_2024_2025.csv"
path_pq_raw  = FINAL_DIR / "cuaca_jabar_12kota_2024_2025.parquet"

# Membaca dari CSV untuk melihat tipe data mentah apa adanya sebelum konversi
print(f"Memuat dataset dari: {path_csv_raw}")
df_raw = pd.read_csv(path_csv_raw)
print("Dimensi awal (baris, kolom):", df_raw.shape)
df_raw.head()

Memuat dataset dari: cuaca-jabar/02_dataset_final/cuaca_jabar_12kota_2024_2025.csv
Dimensi awal (baris, kolom): (210528, 26)


      city status_administratif        region    province  latitude  longitude  elevation_m              time  temperature_2m  relative_humidity_2m  dew_point_2m  precipitation  rain  showers  \
0  Bandung                 Kota  Bandung Raya  Jawa Barat -6.924429  107.60539        703.0  2024-01-01T00:00            20.2                    93          18.9            0.0   0.0      0.0   
1  Bandung                 Kota  Bandung Raya  Jawa Barat -6.924429  107.60539        703.0  2024-01-01T01:00            19.8                    93          18.7            0.0   0.0      0.0   
2  Bandung                 Kota  Bandung Raya  Jawa Barat -6.924429  107.60539        703.0  2024-01-01T02:00            19.8                    92          18.5            0.0   0.0      0.0   
3  Bandung                 Kota  Bandung Raya  Jawa Barat -6.924429  107.60539        703.0  2024-01-01T03:00            20.0                    90          18.3            0.0   0.0      0.0   
4  Bandung               

---
## 3. Tahap A — Data Assessing (Diagnosis Kualitas Data)

Pada bagian ini, variabel `df_raw` **tidak dimodifikasi sama sekali**. Seluruh pemeriksaan bertujuan mendokumentasikan bukti kuantitatif mengenai masalah kualitas data yang ada.

### 3.1 Pemeriksaan Struktur & Tipe Data Semantik

In [3]:
df_raw.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210528 entries, 0 to 210527
Data columns (total 26 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   city                       210528 non-null  object 
 1   status_administratif       210528 non-null  object 
 2   region                     210528 non-null  object 
 3   province                   210528 non-null  object 
 4   latitude                   210528 non-null  float64
 5   longitude                  210528 non-null  float64
 6   elevation_m                210528 non-null  float64
 7   time                       210528 non-null  object 
 8   temperature_2m             210528 non-null  float64
 9   relative_humidity_2m       210528 non-null  int64  
 10  dew_point_2m               210528 non-null  float64
 11  precipitation              210528 non-null  float64
 12  rain                       210528 non-null  float64
 13  showers                    21

**Temuan Tipe Data Semantik (3.1):**
1. **`time` masih bertipe `object` (string ISO 8601 `"YYYY-MM-DDTHH:MM"`):** Bahkan di dalam berkas `.parquet` hasil *gathering*, kolom `time` masih tersimpan sebagai string UTF-8. Kolom ini wajib dikonversi ke `datetime64[ns]` agar ekstraksi komponen waktu (jam, bulan, musiman) dan operasi *rolling window* dapat berjalan.
2. **`weather_code` bertipe `int64`:** Kode WMO (`0, 1, 2, 3, ..., 95`) adalah **label kategorik** kondisi cuaca, bukan besaran kuantitatif kontinu (kode `80` bukan berarti 26 kali lebih besar dari kode `3`). Membiarkannya sebagai `int64` berisiko membuat model regresi/jarak memperlakukannya sebagai angka kontinu.
3. **`city`, `status_administratif`, `region` bertipe `object`:** Ketiga kolom teks ini memiliki kardinalitas sangat rendah (2 hingga 12 nilai unik dari 210.528 baris) sehingga jauh lebih efisien bila dikonversi ke tipe `category`.

---

### 3.2 Audit Kelengkapan (*Missing Values*) & Pola Distribusi Temporalnya

Kita periksa jumlah nilai kosong (`NaN`) di seluruh kolom, lalu menelusuri apakah kemunculan nilai kosong tersebut merata di semua kota dan kapan rentang waktu terjadinya.

In [4]:
null_summary = pd.DataFrame({
    "jumlah_kosong": df_raw.isna().sum(),
    "persen_kosong": (df_raw.isna().mean() * 100).round(2),
})
print("=== Kolom dengan Missing Values ===")
print(null_summary[null_summary["jumlah_kosong"] > 0])

mask_null_prob = df_raw["precipitation_probability"].isna()
print("\n=== Distribusi Missing Values precipitation_probability per Kota ===")
print(df_raw[mask_null_prob].groupby("city").size())

tahun_series = df_raw["time"].str.slice(0, 4)
print("\n=== Distribusi Missing Values per Tahun ===")
print(df_raw["precipitation_probability"].isna().groupby(tahun_series).sum())

print("\nCap waktu NaN pertama :", df_raw.loc[mask_null_prob, "time"].min())
print("Cap waktu NaN terakhir:", df_raw.loc[mask_null_prob, "time"].max())

=== Kolom dengan Missing Values ===
                           jumlah_kosong  persen_kosong
precipitation_probability          34128          16.21

=== Distribusi Missing Values precipitation_probability per Kota ===
city
Bandung        2844
Banjar         2844
Bekasi         2844
Bogor          2844
Cianjur        2844
Cimahi         2844
Cirebon        2844
Depok          2844
Garut          2844
Purwakarta     2844
Sukabumi       2844
Tasikmalaya    2844
Name: count, dtype: int64

=== Distribusi Missing Values per Tahun ===
tahun
2024    34128
2025        0
Name: precipitation_probability, dtype: int64

Cap waktu NaN pertama : 2024-01-01T00:00
Cap waktu NaN terakhir: 2024-12-06T21:00


**Temuan Missing Values (3.2):**
- Dari 26 kolom, **hanya 1 kolom** yang memiliki nilai kosong, yaitu `precipitation_probability` sebanyak **34.128 baris (16,21%)**. Ke-25 kolom lainnya **100% lengkap (0 nilai kosong)**.
- Jumlah nilai kosong **identik di setiap kota**, yaitu tepat **2.844 jam per kota** ($12 \times 2.844 = 34.128$), dan **100% terjadi di tahun 2024** (antara `2024-01-01T00:00` hingga `2024-12-06T21:00`), sedangkan tahun 2025 terisi penuh tanpa satu pun `NaN`.
- **Diagnosis Mekanisme Missingness:** Pola ini bukan *Missing Completely at Random (MCAR)*, melainkan *Structural Archive Missingness* karena arsip model prakiraan Open-Meteo pada paruh awal 2024 belum mengarsipkan variabel probabilitas presipitasi secara kontinu. Namun, karena seluruh variabel cuaca fisik pada jam yang sama (`rain`, `showers`, `weather_code`, `cloud_cover`, `relative_humidity_2m`) **tersedia 100% lengkap**, nilai `precipitation_probability` yang kosong dapat diimputasi secara akurat menggunakan distribusi empiris berdasarkan kelompok `(weather_code, status_hujan)`.

---

### 3.3 Audit Variansi, Duplikasi Kunci, & Kolom Konstan (*Zero-Variance Features*)

In [5]:
print("Duplikat kunci (city, time):", df_raw.duplicated(subset=["city", "time"]).sum())
print("Duplikat baris penuh       :", df_raw.duplicated().sum())

variansi_df = pd.DataFrame({
    "tipe_data": df_raw.dtypes.astype(str),
    "nilai_unik": df_raw.nunique(dropna=True),
}).sort_values("nilai_unik")

variansi_df["status_variansi"] = np.where(
    variansi_df["nilai_unik"] <= 1,
    "KONSTAN (Zero Variance)",
    "Informatif",
)

print("\n=== Kardinalitas (Jumlah Nilai Unik) Seluruh Kolom ===")
print(variansi_df)

Duplikat kunci (city, time): 0
Duplikat baris penuh       : 0

=== Kardinalitas (Jumlah Nilai Unik) Seluruh Kolom ===
                      tipe_data  nilai_unik             status_variansi
province                 object           1  KONSTAN (Zero Variance)
snowfall                float64           1  KONSTAN (Zero Variance)
snow_depth              float64           1  KONSTAN (Zero Variance)
status_administratif     object           2                   Informatif
region                   object           7                   Informatif
latitude                float64           8                   Informatif
city                     object          12                   Informatif
longitude               float64          12                   Informatif
elevation_m             float64          12                   Informatif
weather_code              int64          15                   Informatif
relative_humidity_2m      int64          81                   Informatif
precipitation_proba

**Temuan Kolom Konstan (3.3):**
Terdapat **3 kolom bernilai tunggal (`nilai_unik == 1`)** di seluruh 210.528 baris:
1. `snowfall` = `0.0` cm (iklim tropis Jawa Barat tidak memiliki presipitasi salju).
2. `snow_depth` = `0.0` m (ketebalan salju selalu nol).
3. `province` = `"Jawa Barat"` (seluruh 12 kota berada di provinsi yang sama).

Ketiga kolom ini memiliki variansi nol ($\sigma^2 = 0$), tidak memberi daya pembeda (*discriminative power*) bagi analisis maupun model Machine Learning, sehingga layak dihapus (*dropped*) pada tahap *Data Cleaning*.

---

### 3.4 Audit Konsistensi Fisik & Domain Meteorologi (*Cross-Column Validation*)

Pemeriksaan kualitas data tingkat lanjut tidak cukup hanya memeriksa satu kolom secara terpisah, melainkan harus menguji **konsistensi fisik antar-kolom**:
1. **Hukum Kekekalan Presipitasi:** Menurut definisi Open-Meteo, `precipitation` adalah total presipitasi (`rain + showers + snowfall`). Karena `snowfall == 0`, maka secara matematis wajib berlaku:
   $$\text{precipitation} = \text{rain} + \text{showers}$$
2. **Hukum Termodinamika Titik Embun:** Titik embun (`dew_point_2m`) tidak boleh melebihi suhu udara aktual (`temperature_2m`). Nilai maksimumnya adalah sama dengan suhu udara ketika kelembapan relatif mencapai 100%:
   $$\text{dew\_point\_2m} \le \text{temperature\_2m}$$
3. **Batas Fisik Variabel (*Physical Range Bounds*):**
   - `relative_humidity_2m`, `cloud_cover`, `precipitation_probability` $\in [0, 100]$
   - `precipitation`, `rain`, `showers`, `wind_speed_10m`, `visibility`, `vapour_pressure_deficit` $\ge 0$
   - `surface_pressure` $\in [850, 1050]$ hPa

In [6]:
 print("=== 1. Audit Konsistensi Hidrologi (precipitation vs rain + showers) ===")
total_hujan_aktual = (df_raw["rain"] + df_raw["showers"]).round(2)
selisih_precip = (df_raw["precipitation"] - total_hujan_aktual).abs()

mask_mismatch = selisih_precip > 0.05
mask_zero_but_showers = (df_raw["precipitation"] == 0) & (df_raw["showers"] > 0)
mask_rain_only = ((df_raw["precipitation"] - df_raw["rain"]).abs() < 0.01) & (df_raw["showers"] > 0)
mask_consistent_sum = ((df_raw["precipitation"] - total_hujan_aktual).abs() < 0.01) & (df_raw["showers"] > 0)

print(f"Baris dengan precipitation != rain + showers (selisih > 0.05 mm): {mask_mismatch.sum():,} ({mask_mismatch.mean()*100:.2f}%)")
print(f"  - Baris di mana precipitation == 0.0 padahal showers > 0      : {mask_zero_but_showers.sum():,}")
print(f"  - Baris di mana precipitation == rain saja saat showers > 0   : {mask_rain_only.sum():,}")
print(f"Rentang waktu ketidaksesuaian: {df_raw.loc[mask_mismatch, 'time'].min()}  s.d.  {df_raw.loc[mask_mismatch, 'time'].max()}")
print(f"Mulai konsisten penuh (precip == rain + showers): {df_raw.loc[mask_consistent_sum, 'time'].min()}  s.d.  {df_raw.loc[mask_consistent_sum, 'time'].max()}")

print("\nDistribusi baris precipitation != rain + showers per Kota:")
print(df_raw[mask_mismatch].groupby("city").size())

print("\n=== 2. Audit Konsistensi Termodinamika (dew_point_2m <= temperature_2m) ===")
mask_dew_anomali = df_raw["dew_point_2m"] > (df_raw["temperature_2m"] + 1e-4)
print(f"Baris dengan dew_point_2m > temperature_2m: {mask_dew_anomali.sum()}")

print("\n=== 3. Audit Batas Fisik Variabel (Out-of-Bounds Check) ===")
oob_rh = ((df_raw["relative_humidity_2m"] < 0) | (df_raw["relative_humidity_2m"] > 100)).sum()
oob_cc = ((df_raw["cloud_cover"] < 0) | (df_raw["cloud_cover"] > 100)).sum()
oob_neg = (df_raw[["precipitation", "rain", "showers", "wind_speed_10m", "visibility", "vapour_pressure_deficit"]] < 0).any(axis=1).sum()
print(f"Pelanggaran relative_humidity_2m di luar [0, 100]: {oob_rh}")
print(f"Pelanggaran cloud_cover di luar [0, 100]         : {oob_cc}")
print(f"Pelanggaran nilai negatif (hujan/angin/VPD/vis)  : {oob_neg}")

=== 1. Audit Konsistensi Hidrologi (precipitation vs rain + showers) ===
Baris dengan precipitation != rain + showers (selisih > 0.05 mm): 41,223 (19.58%)
  - Baris di mana precipitation == 0.0 padahal showers > 0      : 13,424
  - Baris di mana precipitation == rain saja saat showers > 0   : 41,219
Rentang waktu ketidaksesuaian: 2024-01-01T10:00  s.d.  2025-12-27T11:00
Mulai konsisten penuh (precip == rain + showers): 2025-09-30T09:00  s.d.  2025-12-31T23:00

Distribusi baris precipitation != rain + showers per Kota:
city
Bandung        2352
Banjar         3906
Bekasi         4208
Bogor          3923
Cianjur        3722
Cimahi         3048
Cirebon        2985
Depok          4207
Garut          2342
Purwakarta     3980
Sukabumi       3218
Tasikmalaya    3332
Name: count, dtype: int64

=== 2. Audit Konsistensi Termodinamika (dew_point_2m <= temperature_2m) ===
Baris dengan dew_point_2m > temperature_2m: 11

=== 3. Audit Batas Fisik Variabel (Out-of-Bounds Check) ===
Pelanggaran relative

> [!IMPORTANT]
> **Temuan Kritis Domain (3.4) — Pergeseran Definisi `precipitation` pada Arsip Model Open-Meteo:**
> - Ditemukan **41.223 baris (19,58% dari seluruh dataset)** di mana nilai `precipitation` **tidak menjumlahkan `showers` (hujan konvektif)**, melainkan hanya menyalin nilai `rain` (hujan stratiform).
> - Akibatnya, terdapat **13.424 jam pengamatan** di mana terjadi hujan konvektif lokal (`showers > 0`), tetapi kolom `precipitation` justru tercatat **`0.0` mm**! Baru mulai `2025-09-30`, arsip model Open-Meteo menjumlahkan `precipitation = rain + showers` secara konsisten.
> - **Implikasi bagi Proyek Mitigasi Banjir:** Hujan konvektif (*showers*) berintensitas tinggi adalah pemicu utama banjir genangan dan banjir bandang di Jawa Barat. Jika ke-41.223 baris ini tidak direkonsiliasi pada tahap *Data Cleaning* (`precipitation = rain + showers`), maka analisis EDA maupun model prediksi curah hujan akan kehilangan hampir 20% kejadian hujan konvektif selama Januari 2024 – September 2025.
> - Selain itu, ditemukan **11 baris** anomali pembulatan numerik di mana `dew_point_2m` melebihi `temperature_2m` sebesar `0.1 °C` saat kelembapan udara mendekati 100%, yang akan dikoreksi dengan *clipping* (`dew_point_2m = np.minimum(dew_point_2m, temperature_2m)`).

---

### 3.5 Audit Spasial (*Grid Snapping*) & Distribusi Kode Cuaca WMO (`weather_code`)

In [7]:
print("=== Koordinat Grid Model Open-Meteo per Kota ===")
grid_kota = (
    df_raw[["city", "region", "status_administratif", "latitude", "longitude", "elevation_m"]]
    .drop_duplicates()
    .sort_values(["latitude", "city"])
    .set_index("city")
)
print(grid_kota)
print(f"\nJumlah nilai unik latitude   : {df_raw['latitude'].nunique()}")
print(f"Jumlah nilai unik longitude  : {df_raw['longitude'].nunique()}")
print(f"Jumlah pasangan (lat,lon) unik: {df_raw[['latitude', 'longitude']].drop_duplicates().shape[0]}")

print("\n=== Distribusi 15 Kode Cuaca WMO (weather_code) ===")
print(df_raw["weather_code"].value_counts().sort_index())

=== Koordinat Grid Model Open-Meteo per Kota ===
             region status_administratif  latitude   longitude  elevation_m
city                                                                       
Bandung        Bandung Raya                 Kota -6.924429  107.605390        703.0
Sukabumi Jawa Barat Selatan                 Kota -6.924429  106.922424        578.0
Cianjur   Jawa Barat Tengah            Kabupaten -6.854130  107.135635        453.0
Cimahi         Bandung Raya                 Kota -6.854130  107.514750        767.0
Bogor               Bodebek                 Kota -6.572935  106.851390        237.0
Purwakarta        Purwasuka            Kabupaten -6.572935  107.455925         88.0
Banjar       Priangan Timur                 Kota -7.346221  108.533900         37.0
Tasikmalaya  Priangan Timur                 Kota -7.346221  108.228810        355.0
Bekasi              Bodebek                 Kota -6.221441  107.006680         19.0
Cirebon             Pantura                

**Temuan Spasial & Kategori WMO (3.5):**
1. **Fenomena *Grid Snapping* pada `latitude`:** Karena model cuaca numerik Open-Meteo bekerja pada grid resolusi ~0,1° (~11 km), empat pasang kota yang berada pada garis lintang berdekatan dipetakan ke baris lintang grid yang sama (`Bandung & Sukabumi: -6.924429`, `Cianjur & Cimahi: -6.854130`, `Bogor & Purwakarta: -6.572935`, `Banjar & Tasikmalaya: -7.346221`). Namun, karena ke-12 nilai `longitude` dan `elevation_m` berbeda, **ke-12 titik grid tetap 100% unik dan terpisah**.
2. **15 Kategori `weather_code`:** Terdapat 15 kode standar WMO yang muncul di Jawa Barat, didominasi oleh kondisi `3` (Berawan Tebal / *Overcast*: 94.687 jam) dan `51` (Gerimis Ringan / *Light Drizzle*: 44.875 jam), hingga cuaca ekstrem `82` (*Violent Rain Showers*: 81 jam) dan `95` (Badai Petir / *Thunderstorm*: 506 jam).

---
### 3.6 Audit Pencilan (*Outliers* — Metode IQR vs. Kejadian Cuaca Ekstrem Nyata)

Pada tahap *assessing*, kita menghitung jumlah observasi yang berada di luar pagar kuartil ($Q_1 - 1.5 \times \text{IQR}$ dan $Q_3 + 1.5 \times \text{IQR}$) untuk seluruh variabel numerik cuaca.

In [8]:
kolom_numerik = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m",
    "precipitation", "rain", "showers", "precipitation_probability",
    "surface_pressure", "cloud_cover", "visibility",
    "vapour_pressure_deficit", "wind_speed_10m",
    "soil_temperature_18cm", "soil_temperature_6cm", "soil_temperature_0cm",
]

baris_outlier = []
for kol in kolom_numerik:
    s = df_raw[kol].dropna()
    q1, med, q3 = s.quantile([0.25, 0.50, 0.75])
    iqr = q3 - q1
    b_bawah = q1 - 1.5 * iqr
    b_atas  = q3 + 1.5 * iqr
    n_bawah = int((s < b_bawah).sum())
    n_atas  = int((s > b_atas).sum())
    baris_outlier.append({
        "kolom": kol,
        "min": round(s.min(), 2),
        "Q1": round(q1, 2),
        "median": round(med, 2),
        "Q3": round(q3, 2),
        "max": round(s.max(), 2),
        "IQR": round(iqr, 2),
        "batas_bawah": round(b_bawah, 2),
        "batas_atas": round(b_atas, 2),
        "outlier_bawah": n_bawah,
        "outlier_atas": n_atas,
        "persen_outlier": round((n_bawah + n_atas) * 100 / len(s), 2),
    })

df_outlier_audit = pd.DataFrame(baris_outlier).set_index("kolom")
print("=== Profil Distribusi & Deteksi Pencilan (Metode 1.5 x IQR) ===")
print(df_outlier_audit)

=== Profil Distribusi & Deteksi Pencilan (Metode 1.5 x IQR) ===
                              min      Q1  median      Q3      max     IQR  batas_bawah  batas_atas  outlier_bawah  outlier_atas  persen_outlier
temperature_2m              13.60   22.50   24.70   27.20    38.60    4.70        15.45       34.25            102          1269            0.65
relative_humidity_2m        19.00   73.00   86.00   94.00   100.00   21.00        41.50      125.50           2534             0            1.20
dew_point_2m                 4.90   20.10   22.00   23.10    28.00    3.00        15.60       27.60           2961             5            1.41
precipitation                0.00    0.00    0.00    0.10    29.00    0.10        -0.15        0.25              0         36255           17.22
rain                         0.00    0.00    0.00    0.10    29.00    0.10        -0.15        0.25              0         34236           16.26
showers                      0.00    0.00    0.00    0.00    14.90

**Analisis Domain terhadap Pencilan (*Outliers* — 3.6):**
1. **Distribusi *Zero-Inflated & Right-Skewed* pada Curah Hujan (`precipitation`, `rain`, `showers`):**
   Karena sebagian besar jam dalam setahun tidak hujan ($Q_1 = 0.0$, $\text{median} = 0.0$, $Q_3 = 0.1$ mm), rumus mekanis $Q_3 + 1.5 \times \text{IQR}$ menetapkan batas atas hanya sebesar **`0.25 mm/jam`**. Akibatnya, setiap hujan di atas `0.25 mm/jam` (17,22% data) terdeteksi sebagai "outlier" statistik. Padahal curah hujan hingga `29.0 mm/jam` adalah **kejadian hujan lebat tropis yang valid** dan justru merupakan **target utama sistem peringatan dini banjir**. Menghapus atau melakukan *capping* IQR pada variabel hujan akan menghancurkan kemampuan model mendeteksi banjir!
2. **Rentang Fisik Seluruh Nilai Ekstrem Valid untuk Iklim Jawa Barat:**
   - Suhu udara `13.6 °C` (malam/dini hari di dataran tinggi Garut/Cimahi/Bandung) hingga `38.6 °C` (siang terik di pesisir Pantura Cirebon/Bekasi) berada dalam rentang klimatologis nyata.
   - Suhu permukaan tanah langsung (`soil_temperature_0cm`) mencapai `48.7 °C` saat radiasi matahari siang puncak di dataran rendah dan turun ke `9.4 °C` pada dini hari di pegunungan, sedangkan lapisan tanah dalam (`18cm`) jauh lebih stabil (`20.1 °C – 35.3 °C`).
   - Defisit tekanan uap (`vapour_pressure_deficit`) hingga `4.94 kPa` pada siang hari kering di Pantura adalah indikator nyata **stres air ekstrem tanaman** untuk kasus penggunaan agrikultur.
- **Keputusan Cleaning:** Seluruh nilai ekstrem meteorologis yang valid secara fisik **dipertahankan 100%** (tidak dihapus/di-*winsorize*), sementara anomali yang melanggar hukum fisika (`dew_point_2m > temperature_2m` dan `precipitation < rain + showers`) diperbaiki secara eksak.

---

### 3.7 Matriks Ringkasan Temuan *Data Assessing* & Rencana *Data Cleaning*

| ID | Kategori Masalah | Kolom Terdampak | Bukti Kuantitatif Hasil Assessing | Tindakan pada Tahap Cleaning |
| :-: | --- | --- | --- | --- |
| **F-01** | **Kolom Konstan (*Zero Variance*)** | `snowfall`, `snow_depth`, `province` | `nunique == 1` di seluruh 210.528 baris (`0.0`, `0.0`, `"Jawa Barat"`). | **Drop 3 kolom** untuk menghilangkan fitur redundan tanpa informasi. |
| **F-02** | **Inkonsistensi Fisik Presipitasi** | `precipitation` terhadap `rain + showers` | **41.223 baris (19,58%)** memiliki `precipitation < rain + showers` (termasuk 13.424 baris `precipitation == 0` saat `showers > 0` sebelum 30 Sep 2025). | **Rekonsiliasi fisik:** Tetapkan `precipitation = (rain + showers).round(2)` di seluruh baris. |
| **F-03** | **Anomali Termodinamika Titik Embun** | `dew_point_2m` terhadap `temperature_2m` | **11 baris** memiliki `dew_point_2m > temperature_2m` (selisih pembulatan `0.1 °C` saat RH ~100%). | **Koreksi *clipping* fisik:** `dew_point_2m = np.minimum(dew_point_2m, temperature_2m)`. |
| **F-04** | **Missing Values (16,21%)** | `precipitation_probability` | **34.128 baris `NaN`** (2.844 baris/kota, semuanya di tahun 2024). | **Imputasi terstratifikasi berbasis domain:** Isi `NaN` dengan median `precipitation_probability` pada kelompok `(weather_code, status_hujan_aktual)`. |
| **F-05** | **Tipe Data Semantik & Label Kategori** | `time`, `weather_code`, `city`, `region`, `status_administratif` | `time` masih `object` (string), `weather_code` masih `int64`, kolom lokasi masih `object`. | Konversi `time` ke `datetime64[ns]`, tambahkan kolom deskripsi `weather_label` (standar WMO), dan ubah kolom kategorik ke tipe `category`. |
| **F-06** | **Pencilan (*Outliers*) Cuaca Ekstrem** | `precipitation`, `showers`, `wind_speed_10m`, `vapour_pressure_deficit` | Ekor distribusi kanan terdeteksi di atas $Q_3 + 1.5 \times \text{IQR}$, namun seluruhnya berada dalam batas fisik klimatologi tropis. | **Pertahankan (*retain*) 100%** karena merupakan sinyal kejadian hujan ekstrem (banjir) dan stres kekeringan tanaman (agrikultur). |

---
## 4. Tahap B — Data Cleaning (Eksekusi Pembersihan & Standarisasi)

Seluruh langkah pembersihan diterapkan pada salinan `df_clean = df_raw.copy()` agar dataset mentah awal tetap terjaga dan setiap langkah transformasi dapat diaudit dampaknya.

### 4.1 Penghapusan Kolom Konstan (`F-01`)

In [9]:
df_clean = df_raw.copy()

kolom_konstan = ["province", "snowfall", "snow_depth"]
df_clean = df_clean.drop(columns=kolom_konstan)

print("Kolom konstan yang dihapus:", kolom_konstan)
print("Dimensi sebelum drop:", df_raw.shape)
print("Dimensi sesudah drop:", df_clean.shape)

Kolom konstan yang dihapus: ['province', 'snowfall', 'snow_depth']
Dimensi sebelum drop: (210528, 26)
Dimensi sesudah drop: (210528, 23)


### 4.2 Rekonsiliasi Fisik Presipitasi (`F-02`) & Koreksi Termodinamika Titik Embun (`F-03`)

1. **Rekonsiliasi `precipitation`:** Menetapkan `precipitation = (rain + showers).round(2)` sehingga definisi total curah hujan konsisten 100% di sepanjang tahun 2024 dan 2025.
2. **Koreksi `dew_point_2m`:** Memastikan `dew_point_2m <= temperature_2m` pada ke-11 baris yang mengalami selisih pembulatan.

In [10]:
precip_sebelum = df_clean["precipitation"].copy()
precip_rekonsiliasi = (df_clean["rain"] + df_clean["showers"]).round(2)

jml_perbaikan_precip = int(((precip_sebelum - precip_rekonsiliasi).abs() > 1e-3).sum())
df_clean["precipitation"] = precip_rekonsiliasi

jml_perbaikan_dew = int((df_clean["dew_point_2m"] > df_clean["temperature_2m"]).sum())
df_clean["dew_point_2m"] = np.minimum(df_clean["dew_point_2m"], df_clean["temperature_2m"])

print(f"Baris precipitation yang direkonsiliasi : {jml_perbaikan_precip:,}")
print(f"Baris dew_point_2m yang dikoreksi       : {jml_perbaikan_dew:,}\n")

perbandingan_precip = pd.DataFrame({
    "Sebelum (Raw)": precip_sebelum.describe(),
    "Sesudah (Reconciled)": df_clean["precipitation"].describe(),
}).T.round(2)
print("Perbandingan Statistik precipitation Sebelum vs. Sesudah Rekonsiliasi:")
print(perbandingan_precip)

Baris precipitation yang direkonsiliasi : 41,223
Baris dew_point_2m yang dikoreksi       : 11

Perbandingan Statistik precipitation Sebelum vs. Sesudah Rekonsiliasi:
                        count  mean   std  min  25%  50%  75%   max
Sebelum (Raw)        210528.0  0.33  1.23  0.0  0.0  0.0  0.1  29.0
Sesudah (Reconciled) 210528.0  0.51  1.53  0.0  0.0  0.0  0.2  29.3


### 4.3 Penanganan *Missing Values* `precipitation_probability` Berbasis Stratifikasi Domain (`F-04`)

Mengisi 34.128 baris kosong pada `precipitation_probability` dengan satu angka rata-rata/median global (`24.0%`) akan menimbulkan distorsi serius: jam cerah tanpa awan (`weather_code == 0`) akan diberi peluang hujan 24%, sementara jam hujan lebat (`weather_code == 65`) juga hanya diberi peluang 24%.

Karena `weather_code` dan status keterjadian hujan aktual (`is_raining = precipitation > 0`) tersedia lengkap pada seluruh 210.528 baris, kita melakukan **Imputasi Median Terstratifikasi (*Domain-Stratified Median Imputation*)** berdasarkan pasangan `(weather_code, is_raining)`:

In [11]:
is_raining = df_clean["precipitation"] > 0.0

# Hitung median referensi dari 176.400 baris yang memiliki nilai precipitation_probability
median_kelompok = (
    df_clean.groupby(["weather_code", is_raining.rename("is_raining")])["precipitation_probability"]
    .median()
    .round(1)
)
print("=== Tabel Referensi Median precipitation_probability per (weather_code, is_raining) ===")
print(median_kelompok)

nan_sebelum = int(df_clean["precipitation_probability"].isna().sum())

# Imputasi berdasarkan kelompok (weather_code, is_raining)
df_clean["precipitation_probability"] = (
    df_clean["precipitation_probability"]
    .fillna(
        df_clean.groupby(["weather_code", is_raining])["precipitation_probability"]
        .transform("median")
        .round(1)
    )
)

nan_sesudah = int(df_clean["precipitation_probability"].isna().sum())
print(f"\nJumlah NaN sebelum imputasi: {nan_sebelum:,}")
print(f"Jumlah NaN sesudah imputasi: {nan_sesudah:,}\n")
print("Statistik precipitation_probability Sesudah Imputasi:")
print(df_clean["precipitation_probability"].describe().round(2))

=== Tabel Referensi Median precipitation_probability per (weather_code, is_raining) ===
weather_code  is_raining
0             False          0.0
              True          20.5
1             False          4.0
              True          33.0
2             False          5.0
              True          41.0
3             False         14.0
              True          50.0
45            False         12.0
51            True          70.0
53            True          85.0
55            True          88.0
61            True          89.0
63            True          90.0
65            True          89.0
80            True          93.0
81            True          95.0
82            True          94.0
95            True          89.0
Name: precipitation_probability, dtype: float64

Jumlah NaN sebelum imputasi: 34,128
Jumlah NaN sesudah imputasi: 0

Statistik precipitation_probability Sesudah Imputasi:
count    210528.00
mean         38.01
std          34.29
min           0.00
25%          

### 4.4 Standarisasi Tipe Data Semantik & Pemetaan Deskripsi Cuaca WMO (`F-05`)

Pada langkah ini dilakukan:
1. Konversi kolom `time` dari `object` menjadi `datetime64[ns]`.
2. Pemetaan 15 kode numerik `weather_code` menjadi kolom deskripsi standar WMO `weather_label` (ditempatkan tepat setelah `weather_code`) agar interpretasi EDA dan klasifikasi cuaca mudah dibaca.
3. Konversi `city`, `status_administratif`, `region`, `weather_code`, dan `weather_label` menjadi tipe `category` untuk menegaskan semantik kategorik sekaligus memangkas pemakaian memori RAM.

In [12]:
WMO_MAPPING = {
    0:  "Cerah (Clear sky)",
    1:  "Cerah Berawan (Mainly clear)",
    2:  "Berawan Sebagian (Partly cloudy)",
    3:  "Berawan Tebal (Overcast)",
    45: "Kabut (Fog)",
    51: "Gerimis Ringan (Light drizzle)",
    53: "Gerimis Sedang (Moderate drizzle)",
    55: "Gerimis Lebat (Dense drizzle)",
    61: "Hujan Ringan (Slight rain)",
    63: "Hujan Sedang (Moderate rain)",
    65: "Hujan Lebat (Heavy rain)",
    80: "Hujan Konvektif Ringan (Slight rain showers)",
    81: "Hujan Konvektif Sedang (Moderate rain showers)",
    82: "Hujan Konvektif Sangat Lebat (Violent rain showers)",
    95: "Badai Petir (Thunderstorm)",
}

mem_awal = df_raw.memory_usage(deep=True).sum() / 1_048_576

# 1. Konversi time ke datetime64[ns]
df_clean["time"] = pd.to_datetime(df_clean["time"])

# 2. Sisipkan kolom weather_label tepat setelah weather_code
idx_wmo = df_clean.columns.get_loc("weather_code")
label_wmo = df_clean["weather_code"].map(WMO_MAPPING)
if "weather_label" not in df_clean.columns:
    df_clean.insert(idx_wmo + 1, "weather_label", label_wmo)
else:
    df_clean["weather_label"] = label_wmo

# 3. Konversi kolom kategorik ke dtype 'category'
for kol_kat in ["city", "status_administratif", "region", "weather_code", "weather_label"]:
    df_clean[kol_kat] = df_clean[kol_kat].astype("category")

mem_akhir = df_clean.memory_usage(deep=True).sum() / 1_048_576
print(f"Memori sebelum optimasi tipe data: {mem_awal:.2f} MB")
print(f"Memori sesudah optimasi tipe data: {mem_akhir:.2f} MB (Hemat {(1 - mem_akhir/mem_awal)*100:.1f}%)\n")

print("=== Distribusi Pasangan weather_code & weather_label ===")
print(df_clean.groupby(["weather_code", "weather_label"], observed=True).size())

Memori sebelum optimasi tipe data: 41.80 MB
Memori sesudah optimasi tipe data: 30.12 MB (Hemat 27.9%)

=== Distribusi Pasangan weather_code & weather_label ===
weather_code  weather_label                                      
0             Cerah (Clear sky)                                      15582
1             Cerah Berawan (Mainly clear)                           15580
2             Berawan Sebagian (Partly cloudy)                       13329
3             Berawan Tebal (Overcast)                               94687
45            Kabut (Fog)                                              452
51            Gerimis Ringan (Light drizzle)                         44875
53            Gerimis Sedang (Moderate drizzle)                       9864
55            Gerimis Lebat (Dense drizzle)                           2910
61            Hujan Ringan (Slight rain)                              4891
63            Hujan Sedang (Moderate rain)                            4892
65            Hujan Leba

---
## 5. Verifikasi Akhir (*Post-Cleaning Quality Gate*) & Penyimpanan Dataset Bersih

Sebelum menyimpan dataset ke `cuaca-jabar/03_dataset_clean/`, seluruh kriteria kualitas diuji ulang menggunakan `assert` untuk menjamin tidak ada satu pun masalah yang tersisa.

In [13]:
n_baris, n_kolom = df_clean.shape
n_null = int(df_clean.isna().sum().sum())
n_konstan = int((df_clean.nunique() <= 1).sum())
n_duplikat = int(df_clean.duplicated(subset=["city", "time"]).sum())
n_mismatch_precip = int(((df_clean["precipitation"] - (df_clean["rain"] + df_clean["showers"])).abs() > 0.05).sum())
n_anomali_dew = int((df_clean["dew_point_2m"] > df_clean["temperature_2m"] + 1e-4).sum())

assert n_baris == 210_528, f"Jumlah baris berubah: {n_baris}"
assert n_kolom == 24, f"Jumlah kolom tidak sesuai: {n_kolom}"
assert n_null == 0, f"Masih ada nilai kosong: {n_null}"
assert n_konstan == 0, f"Masih ada kolom konstan: {n_konstan}"
assert n_duplikat == 0, f"Masih ada duplikat: {n_duplikat}"
assert n_mismatch_precip == 0, f"Masih ada selisih precip: {n_mismatch_precip}"
assert n_anomali_dew == 0, f"Masih ada anomali dew_point: {n_anomali_dew}"

print("=== POST-CLEANING QUALITY GATE ===")
print(f"[LULUS] Total baris utuh                   : {n_baris:,} baris (12 kota x 17,544 jam)")
print(f"[LULUS] Total kolom bersih                 : {n_kolom} kolom")
print(f"[LULUS] Total missing values (NaN)         : {n_null}")
print(f"[LULUS] Total kolom konstan (nunique <= 1) : {n_konstan}")
print(f"[LULUS] Total duplikat (city, time)        : {n_duplikat}")
print(f"[LULUS] Selisih precip vs (rain + showers) : {n_mismatch_precip} baris")
print(f"[LULUS] Anomali dew_point_2m > temp_2m     : {n_anomali_dew} baris\n")

df_clean.info(show_counts=True)

=== POST-CLEANING QUALITY GATE ===
[LULUS] Total baris utuh                   : 210,528 baris (12 kota x 17,544 jam)
[LULUS] Total kolom bersih                 : 24 kolom
[LULUS] Total missing values (NaN)         : 0
[LULUS] Total kolom konstan (nunique <= 1) : 0
[LULUS] Total duplikat (city, time)        : 0
[LULUS] Selisih precip vs (rain + showers) : 0 baris
[LULUS] Anomali dew_point_2m > temp_2m     : 0 baris

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210528 entries, 0 to 210527
Data columns (total 24 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   city                       210528 non-null  category      
 1   status_administratif       210528 non-null  category      
 2   region                     210528 non-null  category      
 3   latitude                   210528 non-null  float64       
 4   longitude                  210528 non-null  float64       
 5   elevation_m      

In [14]:
path_csv_clean = CLEAN_DIR / "cuaca_jabar_12kota_2024_2025_clean.csv"
path_pq_clean  = CLEAN_DIR / "cuaca_jabar_12kota_2024_2025_clean.parquet"

df_clean.to_csv(path_csv_clean, index=False)
df_clean.to_parquet(path_pq_clean, index=False)

print("DATASET BERSIH HASIL ASSESSING & CLEANING")
print("-" * 60)
print(f"Jumlah baris  : {df_clean.shape[0]:,}")
print(f"Jumlah kolom  : {df_clean.shape[1]}")
print(f"Jumlah lokasi : {df_clean['city'].nunique()}")
print(f"Periode       : {df_clean['time'].min()}  s.d.  {df_clean['time'].max()}")
print(f"CSV Bersih    : {path_csv_clean}  ({path_csv_clean.stat().st_size / 1_048_576:.1f} MB)")
print(f"Parquet Bersih: {path_pq_clean}  ({path_pq_clean.stat().st_size / 1_048_576:.1f} MB)")
print("-" * 60)

df_clean.head(5)

DATASET BERSIH HASIL ASSESSING & CLEANING
------------------------------------------------------------
Jumlah baris  : 210,528
Jumlah kolom  : 24
Jumlah lokasi : 12
Periode       : 2024-01-01 00:00:00  s.d.  2025-12-31 23:00:00
CSV Bersih    : cuaca-jabar/03_dataset_clean/cuaca_jabar_12kota_2024_2025_clean.csv  (34.5 MB)
Parquet Bersih: cuaca-jabar/03_dataset_clean/cuaca_jabar_12kota_2024_2025_clean.parquet  (3.3 MB)
------------------------------------------------------------


      city status_administratif        region  latitude  longitude  elevation_m                time  temperature_2m  relative_humidity_2m  dew_point_2m  precipitation  rain  showers  \
0  Bandung                 Kota  Bandung Raya -6.924429  107.60539        703.0 2024-01-01 00:00:00            20.2                    93          18.9            0.0   0.0      0.0   
1  Bandung                 Kota  Bandung Raya -6.924429  107.60539        703.0 2024-01-01 01:00:00            19.8                    93          18.7            0.0   0.0      0.0   
2  Bandung                 Kota  Bandung Raya -6.924429  107.60539        703.0 2024-01-01 02:00:00            19.8                    92          18.5            0.0   0.0      0.0   
3  Bandung                 Kota  Bandung Raya -6.924429  107.60539        703.0 2024-01-01 03:00:00            20.0                    90          18.3            0.0   0.0      0.0   
4  Bandung                 Kota  Bandung Raya -6.924429  107.60539        7

---
## 6. Ringkasan Tahap Assessing & Cleaning

| Aspek Kualitas Data | Kondisi Awal (`02_dataset_final`) | Tindakan Pembersihan | Kondisi Akhir (`03_dataset_clean`) |
| --- | --- | --- | --- |
| **Dimensi Kolom** | 26 kolom (termasuk 3 kolom konstan `province`, `snowfall`, `snow_depth`) | Hapus 3 kolom konstan + tambah 1 kolom deskripsi `weather_label` | **24 kolom informatif** (0 kolom konstan) |
| **Missing Values** | **34.128 baris kosong (16,21%)** pada `precipitation_probability` | Imputasi median terstratifikasi berdasarkan `(weather_code, is_raining)` | **0 baris kosong (100% lengkap)** |
| **Konsistensi Presipitasi** | **41.223 baris (19,58%)** memiliki `precipitation < rain + showers` | Rekonsiliasi fisik `precipitation = (rain + showers).round(2)` | **100% konsisten (`0` selisih)** |
| **Konsistensi Titik Embun** | **11 baris** memiliki `dew_point_2m > temperature_2m` | Koreksi *clipping* fisik `dew_point_2m = min(dew_point_2m, temperature_2m)` | **100% mematuhi hukum termodinamika** |
| **Tipe Data & Memori** | `time` bertipe `object`, `weather_code` bertipe `int64` (~41,8 MB RAM) | Konversi `time` ke `datetime64[ns]` & 5 kolom kategorik ke `category` | **Tipe semantik tepat (~30,1 MB RAM, hemat 27,9%)** |
| **Pencilan (*Outliers*)** | Nilai hujan & VPD tinggi terdeteksi oleh rumus mekanis $1.5 \times \text{IQR}$ | Diverifikasi valid secara klimatologis; dipertahankan 100% | **Sinyal hujan ekstrem & kekeringan tetap utuh** |

---
**Langkah Berikutnya (Tahap 3 — Exploratory Data Analysis):**  
Dataset bersih di `cuaca-jabar/03_dataset_clean/` siap dianalisis untuk melihat pola spasiotemporal curah hujan ekstrem antar-7 region Jawa Barat serta dinamika VPD dan suhu tanah berlapis pada sektor pertanian.